
# What Drives Airbnb Pricing in Vienna?
### Combining structured listing data with guest review text to model price and understand what guests notice at each price point

**Author:** Olga Nureeva · [LinkedIn](https://linkedin.com/in/olga-nureeva) · [GitHub](https://github.com/Onureeva)

---

**Business question.** Treat a short-term rental as an investment asset rather than a listing. Someone acquiring or repositioning a Vienna property faces two decisions: **which price tier this asset can realistically compete in**, and **what to spend money on to get there**. Both decisions turn on a distinction the structured data alone cannot make — some attributes are bought with the property and fixed at the moment of purchase, while others can be improved afterwards with capital.

Guest review text is where that distinction becomes visible. Reviews record what guests at each price point actually noticed and cared about, which is a different question from what the listing objectively contains. So this project asks two things: whether review text carries pricing signal beyond the structured fields, and — more usefully for an investor — **what each price tier's guests emphasise, and therefore which of those things can be changed and which cannot**.

**Data.** [Inside Airbnb](https://insideairbnb.com/get-the-data/) — Vienna, Austria, snapshot of 14 September 2025. Two sources: `listings.csv.gz` (property attributes and price) and `reviews.csv.gz` (guest review text). After cleaning and removing listings with missing prices, the modelling set is **8,139 unique listings**, each paired with the concatenated text of its 10 most recent reviews.

**Headline results.**

| Model | R² (log price) | MAE | Median AE |
|---|---|---|---|
| Median baseline | — | $79.63 | $30.00 |
| Structural features only | 0.371 | $67.28 | $21.98 |
| Structural + NLP features | **0.402** | **$66.05** | **$21.53** |

Review text adds real but modest predictive value: **+0.03 R², and $1.22 off mean absolute error**. Property attributes remain the dominant driver of price — which is itself the first investment finding: the asset, not its reputation, sets the rate.

Where the text earns its place is in explaining *why*, and that is where the decision value sits. Review topics shift systematically across price tiers (Kruskal–Wallis, Holm-adjusted p < 0.001 for all four), and they split cleanly into what an investor buys and what an investor can build.

**Method.** EDA and cleaning → TF-IDF and n-grams → topic modelling compared across LDA, NMF, and LSA (NMF selected for topic separability) → DistilBERT sentiment scoring → feature engineering → Ridge regression and a feed-forward neural network, each evaluated against a median baseline → price-segment analysis with non-parametric significance testing.

**How to run.** Python 3.10+, dependencies in `requirements.txt`. Listings and reviews download directly from Inside Airbnb; the cached DistilBERT sentiment scores are in `data/listing_sentiment.csv`. Run cells top to bottom. Sentiment scoring is the only expensive step and is cached — the code that produced it is preserved and documented in the sentiment section.

**A note on an error I found and corrected.** An earlier version of this analysis merged listings against individual reviews rather than against per-listing aggregated text. This duplicated each listing up to ten times, so a random train/test split placed the same listing — with the same price and the same structural features — on both sides of the split. Reported R² was 0.62. After correcting the merge to one row per listing, honest R² is 0.402. The lower number is the real one. I have left this note in because catching it mattered more to the quality of the work than the metric did.


# 1. Exploratory Data Analysis

### Importing the required Libraries and the Dataset

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scipy import stats


In [2]:
# import datasets
listings_url = 'https://data.insideairbnb.com/austria/vienna/vienna/2025-09-14/data/listings.csv.gz'

reviews_url = 'https://data.insideairbnb.com/austria/vienna/vienna/2025-09-14/data/reviews.csv.gz'

# Load the datasets into DataFrames
listings_df = pd.read_csv(listings_url, compression='gzip', encoding='mac_roman')

reviews_df = pd.read_csv(reviews_url, compression='gzip')



In [3]:
# Load preprocessed reviews
df_cleaned = pd.read_csv("../data/cleaned_reviews.csv")

print(f"Cleaned reviews shape: {df_cleaned.shape}")
df_cleaned.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/cleaned_reviews.csv'

## Data Cleaning (Reviews Dataset)

In [ ]:
#Please use the link below to access the colab file for the reviews cleaning and analysis.
#This was done separately as the reveiws data loads very slowly.
# https://drive.google.com/file/d/1izE0iuUXv55_1LFI0viDYRIOTUWdJ_Mb/view?usp=sharing


## Data Cleaning (Listing Dataset)

In [ ]:
#Listing the top 5 rows of tthe dataset
listings_df.head()

In [ ]:
#Checking the number of rows and columns in the dataset
listings_df.shape

In [ ]:
# Listing the names of all the columns.
listings_df.columns

In [ ]:
listings_df.info()

In [ ]:
# Checking Duplicate rows
listings_df[listings_df.duplicated()]
# to remove the duplicate columns
# listings_df.drop_duplicates(inplace = True)

In [ ]:
missing_perc = listings_df.isna().sum()/len(listings_df) * 100
# Filtering for values > 0 and sorting them from highest to lowest
missing_perc[missing_perc > 0].sort_values(ascending=False)

Since the number of columns are a lot, we eliminate the the text based columns which have no relation to the analysis.

Since the price Column has around 27% of the missing values, removing the rows would be a better alternative as populating the missing rows with mean/median/mode will not help us in our analysis (the price of each property is different)

In [ ]:
listings_df.dropna(subset=['price'], inplace = True)

In [ ]:
missing_perc1 = listings_df.isna().sum()/len(listings_df) * 100
# Filtering for values > 0 and sorting them from highest to lowest
missing_perc1[missing_perc1 > 0].sort_values(ascending=False)

In [ ]:
listings_df.shape

In [ ]:
# Dropping columns more than 50% of missing values.
# (calendar_updated, neighbourhood_group_cleansed, license, host_neighbourhood, neighborhood_overview, neighbourhood, host_about)
Columns_to_Drop = missing_perc1[missing_perc1 > 40].index.tolist()
listings_df.drop(columns = Columns_to_Drop, inplace = True)

In [ ]:
listings_df.shape

In [ ]:
# Dropping the columns which are irrelavant to our analysis.
Columns_to_Drop1 = ['listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'has_availability',
       'availability_30', 'availability_60', 'availability_90',
       'availability_365', 'calendar_last_scraped','bathrooms_text']

listings_df.drop(columns = Columns_to_Drop1, inplace = True)

In [ ]:
listings_df.shape

In [ ]:
listings_df.isna().mean()*100

In [ ]:
#Dropping the tiny number of rows with missing bed/bath values
listings_df.dropna(subset=['bathrooms'], inplace = True)
listings_df.dropna(subset=['bedrooms'], inplace = True)
listings_df.dropna(subset=['beds'], inplace = True)

In [ ]:
import seaborn as sns

#Converts price to an integer.
listings_df['price'] = listings_df['price'].str.replace(',', '').str.replace('$', '').str.replace('.00', '').astype(int)

#Separate numeric columns
num_cols = ['id', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds','number_of_reviews',
'number_of_reviews_ltm',
'number_of_reviews_l30d',
'availability_eoy',
'number_of_reviews_ly',
'estimated_occupancy_l365d',
'estimated_revenue_l365d',
'review_scores_rating',
'review_scores_accuracy',
'review_scores_cleanliness',
'review_scores_checkin'	,
'review_scores_communication',
'review_scores_location',
'review_scores_value',
'calculated_host_listings_count',
'calculated_host_listings_count_entire_homes',
'calculated_host_listings_count_private_rooms',
'calculated_host_listings_count_shared_rooms',
'reviews_per_month','price',]


#Generating a correlation heatmap
fig, ax = plt.subplots(figsize=(20,20))
sns.heatmap(listings_df[num_cols].corr(), annot=True, cmap='coolwarm', linewidths=.5, ax=ax, fmt='.2f')




The review_scores columns are highly correlated, and reducing them will be important for feature selection. Number_of_reviews, host_listings_count, and accomodates should also be considered.

In [ ]:
#Removing some of the highly correlated columns
Columns_to_Drop2 = ['review_scores_accuracy',
'number_of_reviews_ltm',
'review_scores_cleanliness',
'review_scores_checkin'	,
'review_scores_communication',
'review_scores_location',
'review_scores_value',
'calculated_host_listings_count',
'calculated_host_listings_count_entire_homes',
'number_of_reviews',
'number_of_reviews_l30d',
'number_of_reviews_ly',
'availability_eoy',
'reviews_per_month',
'estimated_occupancy_l365d',
'estimated_revenue_l365d',]

listings_df.drop(columns = Columns_to_Drop2, inplace = True)

In [ ]:
#Checking the shape
print(listings_df.shape)
print("------------")
# Checking percent of missing values for column.
missing_perc1 = listings_df.isna().sum()/len(listings_df) * 100
print(missing_perc1)

In [ ]:
num_cols = [
    'id', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'price', 'review_scores_rating', 'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms'
]

#Histogram of numeric columns
listings_df[num_cols].hist(bins=15, figsize=(20,8))
plt.tight_layout()
plt.show()

In [ ]:
listings_df[num_cols].describe()


In [ ]:
#Importing cleaned reviews
import requests
from io import StringIO

cleaned_reviews_link = "https://drive.google.com/drive/u/3/folders/1sjDP_CfyAjRmIT1dzmJonMDu3ZCgVGfv"

file_id = cleaned_reviews_link.split('/')[-2]
dwn_url='https://drive.google.com/uc?export=download&id=' + file_id
url = requests.get(dwn_url).text

csv_raw = StringIO(url)
cleaned_reviews_df = pd.read_csv(csv_raw)
print(cleaned_reviews_df.head())

# Filtering By Date

In [ ]:
df_cleaned.head()
df_cleaned.shape

In [ ]:
#Filtering for the last 10 reviews per listing, and concatenating them

print('Pre-sort')
print(df_cleaned.nunique())



# Code from class tutorial
# Get the last 10 reviews per listing
latest_reviews = (
    df_cleaned.sort_values(['listing_id', 'date'], ascending=[True, False])
      .groupby('listing_id')
      .head(10)
)



# Concatenate all reviews into one combined text per listing
listing_texts = (
  latest_reviews
   .groupby('listing_id')['comments']
   .apply(lambda x: " ".join(x.astype(str)))  # Combine all reviews into one string
   .reset_index(name='combined_reviews')
)





# Check the shape of the new DataFrame
listing_texts.shape

print('Post-sort')
print(listing_texts.nunique())

listing_texts.head()

# Merging the Reviews and Listings Data

In [ ]:
#Merging the two dataframes using the common listing ID, then checking the head and shape.
merged_df = pd.merge(
    listings_df,
    listing_texts,
    left_on='id',
    right_on='listing_id',
    how='inner'
).drop(columns=['id'])

print(merged_df.shape)
print("Unique listings:", merged_df["listing_id"].nunique())
print("Duplicate listing IDs:", merged_df["listing_id"].duplicated().sum())
print(merged_df.head())



In [ ]:
print(merged_df.shape)
print("Unique listings:", merged_df["listing_id"].nunique())
print("Duplicate listing IDs:", merged_df["listing_id"].duplicated().sum())

In [ ]:
merged_df.isnull().sum()

In [ ]:
merged_df.info()

Text Preprocessing Raw reviews often contain punctuation, capitalization, numbers, and stopwords (common words like the, is, at). Before applying NLP techniques such as Bag-of-Words or TF-IDF, we need to clean and standardize the text.

In this step, we will:

Convert text to lowercase Remove punctuation and numbers Remove common stopwords (Optional) Apply stemming or lemmatization — reducing words to their root form (running → run)

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Ensure 'cleaned_comments' column exists. If not, re-create it.
if "cleaned_comments" not in merged_df.columns:
    # Download stopwords & wordnet if not already
    nltk.download("stopwords", quiet=True)
    nltk.download("wordnet", quiet=True)

    stop_words = set(stopwords.words("english"))
    custom_stopwords = {"airbnb", "vienna", "recommend", "apartment", "good","great","nice", "really", "stay", "time","place","city","definitely", "absolutely", "clean", "location"}
    stop_words = stop_words.union(custom_stopwords)
    lemmatizer = WordNetLemmatizer()

    def preprocess_text(text):
        # Lowercase
        text = text.lower()
        # Remove punctuation and numbers
        text = re.sub(r'[^a-z\s]', '', text)
        # Tokenize
        tokens = text.split()
        # Remove stopwords and lemmatize
        tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
        tokens=list(dict.fromkeys(tokens))
        return " ".join(tokens)

    # Apply preprocessing
    merged_df["cleaned_comments"] = (
    merged_df["combined_reviews"]
    .fillna("")
    .apply(preprocess_text)
)

# Combine all cleaned reviews into one big text
all_text = " ".join(merged_df["cleaned_comments"])

# Generate word cloud
wordcloud = WordCloud(width=800, height=400, background_color="white",
                      max_words=100, colormap="viridis").generate(all_text)

# Display
plt.figure(figsize=(12,6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Airbnb Reviews in Vienna", fontsize=16)
plt.show()

Bag-of-Words (Word Counts) The Bag-of-Words model represents text by counting how many times each word appears.

It ignores word order, grammar, and context. Each document becomes a vector of word frequencies. While simple, it often works surprisingly well for tasks like sentiment classification or topic clustering. We will use CountVectorizer from scikit-learn to transform our Airbnb reviews into a document-term matrix.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize vectorizer (limit vocab size for demo)
vectorizer = CountVectorizer(max_features=20, stop_words='english')

# Fit and transform cleaned reviews
bow_matrix = vectorizer.fit_transform(merged_df['cleaned_comments'])

# Convert to DataFrame for readability
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=vectorizer.get_feature_names_out())

print("Shape of BoW matrix:", bow_df.shape)
bow_df.head(10)

TF-IDF (Term Frequency–Inverse Document Frequency) Bag-of-Words gives us raw counts of words, but it does not distinguish between common words (like room, place, stay) and more meaningful words.

TF-IDF improves on this by weighting words based on two factors:

Term Frequency (TF): How often a word appears in a single document. Inverse Document Frequency (IDF): How rare the word is across the entire collection of documents. Why it matters Words that are frequent in a single review but rare across all reviews (e.g., fireplace, mountain, cozy) receive higher importance. Very common words that appear in nearly every review (e.g., room, good, stay) are downweighted. Key Takeaway TF-IDF highlights the most distinctive words in reviews, giving us a more meaningful text representation for analysis and modeling.

We will now apply TF-IDF to our Airbnb sample reviews and inspect the top weighted words.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Apply TF-IDF
tfidf = TfidfVectorizer(max_features=1000, max_df=0.85, min_df=15, stop_words='english', ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(merged_df['cleaned_comments'])

# Convert to DataFrame for inspection
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())

# Show top rows
print("TF-IDF matrix (first 5 rows):")
display(tfidf_df.head())

Words like "comfortable", "great", "host", "location", "station", "city", "recommend" tend to have higher IF-IDF scores in specific listings. This indicates these words are particulary important or distinctive for those reviews, even if they may not appear frequently across all listings.

##N-grams
N-grams capture phrases and context, making them more informative for understanding text compared to isolated words. They are especially useful in sentiment analysis, topic modeling, and search engines where meaning depends on word combinations.

In the next step, we will generate bigrams and trigrams from the Vienna Airbnb reviews and explore the most common ones.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Use bigrams and trigrams (n=2 and n=3)
ngram_vectorizer = CountVectorizer(ngram_range=(2,3), stop_words='english', max_features=30)

# Fit and transform on the cleaned reviews
X_ngrams = ngram_vectorizer.fit_transform(merged_df['cleaned_comments'])

# Convert to dataframe
ngrams_df = pd.DataFrame(X_ngrams.toarray(), columns=ngram_vectorizer.get_feature_names_out())

# Get the top n-grams by summing across all documents
top_ngrams = ngrams_df.sum().sort_values(ascending=False).head(20)

print("Top 20 most frequent bigrams and trigrams:")
print(top_ngrams)

In [ ]:
# Define functions to create plots

import matplotlib.pyplot as plt
import numpy as np
from wordcloud import WordCloud

# Function to plot the top words for each topic
def plot_top_words(model, feature_names, n_top_words=10, title='Top words per topic'):
    fig, axes = plt.subplots(1, model.n_components, figsize=(15, 5), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]
        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        ax.set_title(f'Topic {topic_idx+1}', fontdict={'fontsize': 10})
        ax.invert_yaxis()
        ax.tick_params(axis='both', which='major', labelsize=10)
    plt.suptitle(title, fontsize=12)
    plt.subplots_adjust(top=0.85, wspace=0.3)
    plt.show()


# Function to generate word clouds for each topic
def plot_word_clouds(model, feature_names, n_top_words=30):
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = {feature_names[i]: topic[i] for i in top_features_ind}
        wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(top_features)
        plt.figure()
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title(f'Topic {topic_idx+1} Word Cloud', fontsize=16)
        plt.show()

##Topic modeling
Latent Dirichlet Allocation (LDA) Treats each document as a mixture of topics. Each topic is represented as a distribution of words. Helps discover interpretable themes in text.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation, NMF, TruncatedSVD

In [ ]:
#LDA
# Set the number of topics
n_topics = 4

# Create and fit the LDA model
lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda_model.fit(bow_matrix)
#lda_model.fit(X_tfidf)

# Extract the top words from each topic
terms = vectorizer.get_feature_names_out()
lda_topics = {}

for index, topic in enumerate(lda_model.components_):
    top_words = [terms[i] for i in topic.argsort()[-10:][::-1]]  # Get top 10 words and reverse order
    lda_topics[f"Topic {index+1}"] = top_words

# Print the LDA topics
print("LDA Topics:")
for topic, words in lda_topics.items():
    print(f"{topic}: {', '.join(words)}")

In [ ]:
plot_top_words(lda_model, terms, n_top_words=10, title='Top words per LDA(TFIDF) topic')
plot_word_clouds(lda_model, terms)

2) Non-negative Matrix Factorization (NMF)
Decomposes the TF-IDF matrix into smaller components. Each component (topic) is described by the top keywords. Works well for short and interpretable topics.

In [ ]:
# Create and fit the NMF model
nmf_model = NMF(n_components=n_topics, random_state=42)
nmf_model.fit(X_tfidf)

# Extract the top words from each topic
nmf_topics = {}
terms=tfidf.get_feature_names_out()
for index, topic in enumerate(nmf_model.components_):
    top_words = [terms[i] for i in topic.argsort()[-10:][::-1]]  # Get top 10 words and reverse order
    nmf_topics[f"Topic {index+1}"] = top_words

# Print the NMF topics
print("NMF Topics:")
for topic, words in nmf_topics.items():
    print(f"{topic}: {', '.join(words)}")

In [ ]:
# NMF topic plots

plot_top_words(nmf_model, terms, n_top_words=10, title='Top words per NMF topic')
plot_word_clouds(nmf_model, terms)

##Latent Semantic Analysis (LSA)

Uses Singular Value Decomposition (SVD) to identify underlying structures in

Captures broader word associations but can be harder to interpret.

In [ ]:
#Fit the LSA model
lsa_model = TruncatedSVD(n_components=n_topics, random_state=42)
lsa_model.fit(X_tfidf)


# Extract the top words for each topic
terms = tfidf.get_feature_names_out()
lsa_topics = {}

for index, topic in enumerate(lsa_model.components_):
    top_words = [terms[i] for i in topic.argsort()[-10:][::-1]]  # Get top 10 words and reverse order
    lsa_topics[f"Topic {index+1}"] = top_words

# Print the LSA topics
print("LSA Topics:")
for topic, words in lsa_topics.items():
    print(f"{topic}: {', '.join(words)}")

In [ ]:
# LSA topic plots

plot_top_words(lsa_model, terms, n_top_words=10, title='Top words per NMF topic')
plot_word_clouds(lsa_model, terms)

In [ ]:
#Let's check the topic distance

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Function to plot topic distances using PCA
def plot_topic_distance_pca(model, title='Topic Distance (PCA)'):
    # Get the topic-word matrix (components_) from the model
    topic_word_matrix = model.components_

    # Apply PCA to reduce the topic-word matrix to 2 dimensions
    pca = PCA(n_components=2)
    topic_pca = pca.fit_transform(topic_word_matrix)

    # Plot the topics in 2D space
    plt.figure(figsize=(8, 6))
    plt.scatter(topic_pca[:, 0], topic_pca[:, 1], c='blue', edgecolor='k', s=100)

    # Annotate the topics with text labels
    for i in range(len(topic_pca)):
        plt.text(topic_pca[i, 0], topic_pca[i, 1], f'Topic {i+1}', fontsize=12)

    plt.title(title)
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.show()

In [ ]:
# compare topic distance of all models

# LDA
plot_topic_distance_pca(lda_model, title='LDA Topic Distance (PCA)')
# NMF
plot_topic_distance_pca(nmf_model, title='NMF Topic Distance (PCA)')
# LSA
plot_topic_distance_pca(lsa_model, title='LSA Topic Distance (PCA)')


NMF produced more distinct and interpretable topics, with clearer separation between location, host quality and comfort-related attributes. Topic 1 is Location&Comfort, Topic 2 is Host Quality & Service, Topic 3 is Customer Satisfaction, Topic 4 is Transport Accessibility

In [ ]:
#Assign topic probabilities:
topic_model=nmf_model.transform(X_tfidf)

In [ ]:
listing_topics = pd.DataFrame(
    topic_model,
    columns=[
    "walkability_local",
    "property_condition",
    "transport_access",
    "host_hospitality"
    ]
)

listing_topics["listing_id"] = merged_df["listing_id"].reset_index(drop=True)

listing_topics.head()

In [ ]:
topic_sums = listing_topics[
    ["walkability_local",
    "property_condition",
    "transport_access",
    "host_hospitality"
    ]
].sum()

print(topic_sums)

In [ ]:
#Normilizing topic shares:
topic_distribution=topic_sums/topic_sums.sum()
print(topic_distribution)

In [ ]:
#Overall topic distribution
topic_distribution.plot(kind='bar', title="Overall distribution of Review Topics")
plt.xticks(rotation=30)
plt.show()

##Using pipeline for sentiment

In [ ]:
# import torch
# from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# tokenizer = DistilBertTokenizer.from_pretrained(
#     "distilbert-base-uncased-finetuned-sst-2-english"
# )
# model = DistilBertForSequenceClassification.from_pretrained(
#     "distilbert-base-uncased-finetuned-sst-2-english"
# )

# model.eval()

In [ ]:
# def get_sentiment_batch(texts, batch_size=32):
#     results = []

#     for i in range(0, len(texts), batch_size):
#         batch = texts[i:i+batch_size]

#         inputs = tokenizer(
#             list(batch),
#             return_tensors="pt",
#             truncation=True,
#             padding=True,
#             max_length=512
#         )

#         with torch.no_grad():
#             logits = model(**inputs).logits

#         probs = torch.nn.functional.softmax(logits, dim=1)
#         scores = (probs[:,1] - probs[:,0]).tolist()

#         results.extend(scores)
#     return results

In [ ]:
# texts=merged_df["cleaned_comments"].fillna("").tolist()
# merged_df["sentiment"] = get_sentiment_batch(texts, batch_size=16)

In [ ]:
# listing_sentiment = (
#     merged_df.groupby("listing_id", as_index=False)["sentiment"]
#     .mean()
#     .rename(columns={"sentiment": "avg_sentiment"})
# )

In [ ]:
# listing_sentiment.head()

In [ ]:
# #saving cleaned dataset
# listing_sentiment.to_csv('listing_sentiment.csv', index=False)
# files.download('listing_sentiment.csv')

In [ ]:
#Importing listing sentiment
import requests
from io import StringIO
import gdown

listing_sentiment_link = "https://drive.google.com/file/d/1rwSbtSDqKWiTj-SiK0GIfeC53dsklgAu/view?usp=drive_link"

# Extract file ID from the link
file_id = listing_sentiment_link.split('/')[-2]

# Construct the direct download URL for gdown
url = 'https://drive.google.com/uc?export=download&id=' + file_id

# Use gdown to download the file directly
gdown.download(url, 'listing_sentiment.csv', quiet=False)

# Load the downloaded CSV into a DataFrame
listing_sentiment_df = pd.read_csv('listing_sentiment.csv')
print(listing_sentiment_df.head())

In [ ]:
listing_sentiment_df.info()

In [ ]:
merged_df = pd.merge(merged_df, listing_sentiment_df, on='listing_id',  how="left")
print(merged_df.head())

In [ ]:
merged_df = pd.merge(merged_df, listing_topics, on='listing_id',  how="left")
print(merged_df.head())

In [ ]:
merged_df.info()

In [ ]:
#Generating a correlation heatmap with the undropped columns AND SENTIMENT
num_cols = ['latitude', 'longitude', 'bathrooms', 'bedrooms', 'beds',
'review_scores_rating',
'calculated_host_listings_count_private_rooms',
'calculated_host_listings_count_shared_rooms',
'price','accommodates',
'avg_sentiment',
"walkability_local",
"property_condition",
"transport_access",
"host_hospitality"]

#Generating a correlation heatmap
fig, ax = plt.subplots(figsize=(20,20))
sns.heatmap(merged_df[num_cols].corr(), annot=True, cmap='coolwarm', linewidths=.5, ax=ax, fmt='.2f')


From the correlation heatmap, review length doesn't appear to have any even slightly strong correlations.
However, avg_sentiment has a 0.51 correlation with review_scores_rating, which would be expected as a higher score would come from a more positive sentiment.
#
Additionally, there are two weak, but present negative correlations with availability and the host's calculated number of private listings. A place perceived negatively may get more customers and thus be perceived more negatively, but it's interesting that this appears for the number of private rooms. This shows up slightly more strongly (-0.3) with the reveiw scores rating. **One explanation may be that with more listings, the host gives less attention and care to each, or is less able to communicate effectively with the guests.**


# EDA - Merged Dataset

In [ ]:
merged_df.shape

In [ ]:
merged_df.columns

In [ ]:
merged_df.info()

### Price Distribution

In [ ]:
# Checking the Price Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw price
axes[0].hist(merged_df['price'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (Raw)')
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Count')

# Log transformed price to handle extreme outliers
axes[1].hist(np.log1p(merged_df['price']), bins=60, color='coral', edgecolor='white')
axes[1].set_title('Price Distribution (Log-transformed)')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Price stats:\n{merged_df['price'].describe()}")
print(f"\nListings with price > $500: {(merged_df['price'] > 500).sum()}")

In [ ]:
# Price by Room Type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

room_avg = merged_df.groupby('room_type')['price'].median().sort_values(ascending=False)

axes[0].bar(room_avg.index, room_avg.values, color='steelblue', edgecolor='white')
axes[0].set_title('Median Price by Room Type')
axes[0].set_xlabel('Room Type')
axes[0].set_ylabel('Median Price (USD)')

# Boxplot to show spread
merged_df.boxplot(column='price', by='room_type', ax=axes[1],
                  showfliers=False)  # showfliers=False hides extreme outliers for readability
axes[1].set_title('Price Spread by Room Type')
axes[1].set_xlabel('Room Type')
axes[1].set_ylabel('Price (USD)')
plt.suptitle('')  # removes the default pandas boxplot title

plt.tight_layout()
plt.show()

In [ ]:
#Top Neighbourhoods by Median Price
top_neighbourhoods = (
    merged_df.groupby('neighbourhood_cleansed')['price']
    .median()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 6))
plt.barh(top_neighbourhoods.index[::-1], top_neighbourhoods.values[::-1], color='teal', edgecolor='white')
plt.title('Top 15 Neighbourhoods by Median Listing Price')
plt.xlabel('Median Price (USD)')
plt.tight_layout()
plt.show()

In [ ]:
# Sentiment vs Price
# Using a sample so the plot isn't overloaded with 58,500 points
sample = merged_df.sample(3000, random_state=42)

plt.figure(figsize=(10, 5))
plt.scatter(sample['avg_sentiment'], sample['price'], alpha=0.3, color='purple', edgecolor='none')
plt.title('Average Review Sentiment vs. Listing Price')
plt.xlabel('Average Sentiment Score')
plt.ylabel('Price (USD)')
plt.ylim(0, 600)  # cap y-axis so outliers don't compress the main data
plt.axhline(merged_df['price'].median(), color='red', linestyle='--', label='Median Price')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Correlation between sentiment and price: {merged_df['avg_sentiment'].corr(merged_df['price']):.3f}")

## Feature Engineering

In [ ]:
#Working on a clean copy so we don't mess up merged_df
model_df = merged_df.copy()

#Counting the number of amenities each listing has
model_df['amenity_count'] = model_df['amenities'].apply(
    lambda x: len(x.split(',')) if isinstance(x, str) else 0
)
print(model_df['amenity_count'])

In [ ]:
# Converting instant_bookable (t/f text) to a binary number
model_df['instant_bookable'] = (model_df['instant_bookable'] == 't').astype(int)

In [ ]:
#One-hot encoding categorical text columns
model_df = pd.get_dummies(model_df, columns=['room_type', 'neighbourhood_cleansed', 'property_type'], drop_first=True)


In [ ]:
print(f"Shape after feature engineering: {model_df.shape}")
print(f"\nColumn list (first 30):\n{list(model_df.columns[:30])}")

In [ ]:
#preparing data for modeling
model_df = merged_df.copy()

model_df["amenity_count"] = model_df["amenities"].apply(
    lambda x: len(x.split(",")) if isinstance(x, str) else 0
)

model_df["instant_bookable"] = (
    model_df["instant_bookable"] == "t"
).astype(int)

cols_to_drop_for_model_df = [
    "listing_id",
    "combined_reviews",
    "cleaned_comments",
    "first_review",
    "last_review",
    "amenities"
]

model_df.drop(
    columns=cols_to_drop_for_model_df,
    inplace=True,
    errors="ignore"
)

model_df = pd.get_dummies(
    model_df,
    columns=[
        "room_type",
        "neighbourhood_cleansed",
        "property_type"
    ],
    drop_first=True
)

In [ ]:
print(model_df.shape)
print(model_df.select_dtypes(exclude=np.number).columns.tolist())

## Test Train Split

In [ ]:
# Re-preparing model_df to ensure it's defined and up-to-date
# This code ensures model_df is available, addressing NameError if previous cell's state was lost
model_df = merged_df.copy()

model_df["amenity_count"] = model_df["amenities"].apply(
    lambda x: len(x.split(",")) if isinstance(x, str) else 0
)

model_df["instant_bookable"] = (
    model_df["instant_bookable"] == "t"
).astype(int)

cols_to_drop_for_model_df = [
    "listing_id",
    "combined_reviews",
    "cleaned_comments",
    "first_review",
    "last_review",
    "amenities"
]

model_df.drop(
    columns=cols_to_drop_for_model_df,
    inplace=True,
    errors="ignore"
)

model_df = pd.get_dummies(
    model_df,
    columns=[
        "room_type",
        "neighbourhood_cleansed",
        "property_type"
    ],
    drop_first=True
)

nlp_features = [
    "avg_sentiment",
    "walkability_local",
    "property_condition",
    "transport_access",
    "host_hospitality"
]

structural_features = [
    col for col in model_df.columns
    if col not in nlp_features + ["price", "log_price"]
]

In [ ]:
# Target Variable & Train/Test Split
from sklearn.model_selection import train_test_split

# Log-transforming price to reduce the skewness
model_df['log_price'] = np.log1p(model_df['price'])


In [ ]:
train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.2,
    random_state=42
)

y_train = model_df.loc[train_idx, "log_price"]
y_test = model_df.loc[test_idx, "log_price"]

In [ ]:
X_struct_train = model_df.loc[train_idx, structural_features]
X_struct_test = model_df.loc[test_idx, structural_features]

In [ ]:
all_features = structural_features + nlp_features

X_all_train = model_df.loc[train_idx, all_features]
X_all_test = model_df.loc[test_idx, all_features]

## Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

scaler_struct = StandardScaler()

X_struct_train_scaled = scaler_struct.fit_transform(
    X_struct_train
)

X_struct_test_scaled = scaler_struct.transform(
    X_struct_test
)

ridge_struct = Ridge(alpha=1.0)

ridge_struct.fit(
    X_struct_train_scaled,
    y_train
)

pred_struct_log = ridge_struct.predict(
    X_struct_test_scaled
)

## Linear regression

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    mean_squared_error,
    r2_score
)
#np.expm1 caluculates in $
actual_prices = np.expm1(y_test)

median_train_price = np.median(
    np.expm1(y_train)
)

median_predictions = np.full(
    len(y_test),
    median_train_price
)

baseline_mae = mean_absolute_error(
    actual_prices,
    median_predictions
)

baseline_medae = median_absolute_error(
    actual_prices,
    median_predictions
)

print("Median baseline")
print(f"MAE: ${baseline_mae:.2f}")
print(f"Median absolute error: ${baseline_medae:.2f}")

In [ ]:
scaler_all = StandardScaler()

X_all_train_scaled = scaler_all.fit_transform(
    X_all_train
)

X_all_test_scaled = scaler_all.transform(
    X_all_test
)

ridge_all = Ridge(alpha=1.0)

ridge_all.fit(
    X_all_train_scaled,
    y_train
)

pred_all_log = ridge_all.predict(
    X_all_test_scaled
)

In [ ]:
def evaluate_price_model(
    name,
    y_true_log,
    y_pred_log
):
    actual = np.expm1(y_true_log)
    predicted = np.expm1(y_pred_log)

    return {
        "Model": name,
        "R2_log": r2_score(
            y_true_log,
            y_pred_log
        ),
        "MAE_dollars": mean_absolute_error(
            actual,
            predicted
        ),
        "Median_AE_dollars": median_absolute_error(
            actual,
            predicted
        ),
        "RMSE_dollars": np.sqrt(
            mean_squared_error(actual, predicted)
        )
    }

In [ ]:
results = [
    {
        "Model": "Median baseline",
        "R2_log": np.nan,
        "MAE_dollars": baseline_mae,
        "Median_AE_dollars": baseline_medae,
        "RMSE_dollars": np.sqrt(
            mean_squared_error(
                actual_prices,
                median_predictions
            )
        )
    },
    evaluate_price_model(
        "Structural only",
        y_test,
        pred_struct_log
    ),
    evaluate_price_model(
        "Structural + NLP",
        y_test,
        pred_all_log
    )
]

results_df = pd.DataFrame(results)

display(results_df)

### Model comparison — key takeaway

Adding NLP features from recent guest reviews improved model performance, but only modestly:

- MAE decreased from approximately 67.28 to 66.05 dollars
- Median absolute error decreased from approximately 21.98 to 21.53 dollars
- R² on log-price increased from 0.371 to 0.402

This suggests that guest-review text contains some incremental pricing signal, but most price variation is explained by structured listing characteristics. Reviews may therefore be more valuable for understanding guest expectations and positioning than for direct price prediction.

In [ ]:
nlp_mae_improvement = (
    results_df.loc[
        results_df["Model"] == "Structural only",
        "MAE_dollars"
    ].iloc[0]
    -
    results_df.loc[
        results_df["Model"] == "Structural + NLP",
        "MAE_dollars"
    ].iloc[0]
)

print(
    f"NLP reduced MAE by "
    f"${nlp_mae_improvement:.2f}"
)

In [ ]:
nlp_mae_improvement = (
    results_df.loc[
        results_df["Model"] == "Structural only",
        "MAE_dollars"
    ].iloc[0]
    -
    results_df.loc[
        results_df["Model"] == "Structural + NLP",
        "MAE_dollars"
    ].iloc[0]
)

print(
    f"NLP reduced MAE by "
    f"${nlp_mae_improvement:.2f}"
)

# 6. Price Segment & Guest Expectation Analysis

In [ ]:
segment_df = merged_df.copy()

segment_df["price_segment"] = pd.qcut(
    segment_df["price"],
    q=4,
    labels=[
        "Budget",
        "Mid-range",
        "Premium",
        "Luxury"
    ]
)

segment_df.groupby(
    "price_segment",
    observed=True
)["price"].agg(
    ["count", "min", "median", "max"]
)

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(segment_df["price"], bins=100)
plt.xlim(0,500)

In [ ]:
segment_df["price"].quantile(
    [0.8,0.9,0.95,0.99]
)

In [ ]:
bins = [0, 75, 125, 200, 500, np.inf]

labels = [
    "Budget",
    "Standard",
    "Premium",
    "High-end",
    "Luxury"
]

segment_df["price_segment"] = pd.cut(
    segment_df["price"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
segment_df.groupby(
    "price_segment",
    observed=True
)["price"].agg(["count", "min", "median", "max"])

In [ ]:
topic_cols = [
    "walkability_local",
    "property_condition",
    "transport_access",
    "host_hospitality"
]


topic_by_segment = (
    segment_df
    .groupby(
        "price_segment",
        observed=True
    )[topic_cols]
    .mean()
)

display(topic_by_segment)

In [ ]:
overall = segment_df[topic_cols].mean()

topic_index = (
    topic_by_segment
    .divide(overall)
    *100
)

display(topic_index.round(1))

In [ ]:
segment_df["dominant_topic"] = (
    segment_df[topic_cols]
    .idxmax(axis=1)
)

dominant = pd.crosstab(
    segment_df["price_segment"],
    segment_df["dominant_topic"],
    normalize="index"
)*100

display(dominant.round(1))

In [ ]:
segment_df.groupby(
    "price_segment",
    observed=True
)["avg_sentiment"].mean()

In [ ]:
from scipy.stats import kruskal


kruskal_results = []

for topic in topic_cols:

    groups = [
        group[topic].dropna().values
        for _, group in segment_df.groupby(
            "price_segment",
            observed=True
        )
    ]

    h_stat, p_value = kruskal(*groups)

    kruskal_results.append({
        "topic": topic,
        "H_statistic": h_stat,
        "p_value": p_value
    })

kruskal_df = pd.DataFrame(kruskal_results)

display(kruskal_df)

In [ ]:
from statsmodels.stats.multitest import multipletests

kruskal_df["p_adjusted"] = multipletests(
    kruskal_df["p_value"],
    method="holm"
)[1]

kruskal_df["significant"] = (
    kruskal_df["p_adjusted"] < 0.05
)

display(kruskal_df)

Review topics differ significantly across Airbnb price segments (Kruskal–Wallis, Holm-adjusted p < 0.001 for all topics).

In [ ]:
plt.figure(figsize=(10,6))

for col in topic_index.columns:
    plt.plot(
        topic_index.index,
        topic_index[col],
        marker='o',
        linewidth=3,
        label=col
    )

plt.axhline(
    100,
    color='gray',
    linestyle='--'
)

plt.legend()

## Conclusions and Limitations

### Key Findings

The analysis suggests that guest priorities differ across Airbnb price segments. Topics such as transportation access, property condition, walkability/local area, and host hospitality do not have the same relative importance across price tiers, indicating that guests evaluate listings differently depending on the market segment.

For price prediction, structural listing characteristics remain substantially more informative than review-derived NLP features. Adding the review-topic features improved model performance by approximately 0.03 in R². This indicates that review text provides additional predictive information, but the improvement is modest compared with the explanatory power of listing attributes such as location, room type, capacity, and amenities.

The best-performing model achieved an R² of approximately 0.40. Therefore, the model explains about 40% of the observed variation in listing prices, while roughly 60% remains unexplained by the variables included in this analysis.

### Practical Implications

The results suggest that hosts may benefit from emphasizing different aspects of their listings depending on their price segment. At the same time, review-derived insights should be treated as complementary to, rather than a replacement for, traditional listing characteristics when explaining or predicting Airbnb prices.

### Limitations

- The analysis is based on a single city, so the findings may not generalize to other Airbnb markets.
- The dataset represents one snapshot in time and does not capture seasonal or longitudinal changes in prices and guest preferences.
- The review topics simplify complex review text into a limited number of predefined themes.
- An R² of approximately 0.40 indicates that important determinants of price remain outside the current model, such as temporal demand, events, more detailed location effects, host pricing strategy, and other market conditions.
- The relationships identified in this project are associative and should not be interpreted as causal effects.

Saving datasets for Tableau dashboards

In [ ]:
segment_df.to_csv(
    "airbnb_segment_analysis.csv",
    index=False
)

In [ ]:
topic_tableau = (
    topic_index
    .reset_index()
    .melt(
        id_vars="price_segment",
        var_name="topic",
        value_name="topic_index"
    )
)

In [ ]:
topic_tableau["topic"] = topic_tableau["topic"].replace({
    "walkability_local": "Walkability & Local Amenities",
    "property_condition": "Property Condition",
    "transport_access": "Transport & Accessibility",
    "host_hospitality": "Host & Hospitality"
})

In [ ]:
topic_tableau.head()

In [ ]:
topic_tableau.to_csv(
    "topic_by_price_segment.csv",
    index=False
)

In [ ]:
results_df.to_csv(
    "model_comparison.csv",
    index=False
)